# 05b — Exploración profunda de embeddings

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 5b — Diagnóstico geométrico + analogías con 3 técnicas + ordinalidad completa

## Motivación

En `05_embeddings_analysis.ipynb` probamos la analogía clásica `Argentina − Messi + Francia ≈ ?` y obtuvimos como resultado **selecciones europeas, no Mbappé**. Este notebook investiga por qué falla y cuánto se puede mejorar con tres técnicas:

- **A.** Restricción estricta del espacio de búsqueda (solo jugadores).
- **B.** L2-normalización antes de la aritmética.
- **C.** **3CosMul** (Levy & Goldberg, 2014) — multiplicativo en lugar de aditivo.

Además armamos un **diagnóstico geométrico** (normas por familia, anisotropía intra-familia) para entender la estructura real del espacio aprendido.

## Estructura

1. Setup
2. **Familia 1 — Vecindades puras**: equipos, jugadores top, eventos, stages
3. **Familia 2 — Ordinalidad emergente**: TODOS los buckets numéricos
4. **Familia 3 — Estructura competitiva**: centroides por confederación
5. **Familia 4 — Diagnóstico geométrico**: por qué fallan las analogías
6. **Familia 5 — Tres técnicas de analogía** comparadas
7. **Familia 6 — Visualizaciones globales**
8. Resumen + tabla final

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys, json, pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
FINETUNE_CKPT = CKPT_DIR / 'finetune_10ep' / 'best.pt'

from data.vocabulary import FootballVocab
from models.d10sformer import D10Sformer, D10SformerConfig
from eval.embedding_analysis import (
    get_token_embedding, top_k_neighbours,
    analogy_query, analogy_query_3cosmul,
    family_norm_stats, intra_family_cosine_distribution, cluster_centroid,
    pca_2d, tsne_2d,
)

vocab = FootballVocab.load(VOCAB_PATH)
model_config = D10SformerConfig(
    vocab_size=len(vocab), d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8, dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'), tie_mlm_weights=True,
)
model = D10Sformer(model_config)
ckpt = torch.load(FINETUNE_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✓ Modelo cargado: vocab={len(vocab)}, params={model.num_parameters():,}')

# Listas de tokens por familia
all_team_tokens   = [t for t in vocab.token_to_id if t.startswith('TEAM_')]
all_player_tokens = [t for t in vocab.token_to_id if t.startswith('PLAYER_')]
all_elo_tokens    = sorted([t for t in vocab.token_to_id if t.startswith('ELO_BUCKET_')],
                            key=lambda x: int(x.split('_')[-1]))
all_form_tokens   = ['FORM_VERY_LOW', 'FORM_LOW', 'FORM_MID', 'FORM_HIGH', 'FORM_VERY_HIGH']
all_form_tokens   = [t for t in all_form_tokens if vocab.has(t)]
all_goals_tokens  = ['GOALS_VERY_LOW', 'GOALS_LOW', 'GOALS_MID', 'GOALS_HIGH', 'GOALS_VERY_HIGH']
all_goals_tokens  = [t for t in all_goals_tokens if vocab.has(t)]
all_event_tokens  = [t for t in vocab.token_to_id if t.startswith('EVENT_')]
all_stage_tokens  = [t for t in vocab.token_to_id if t.startswith('STAGE_')]

print(f'\nFamilias de tokens:')
print(f'  TEAM_*:   {len(all_team_tokens)}')
print(f'  PLAYER_*: {len(all_player_tokens)}')
print(f'  ELO_*:    {len(all_elo_tokens)} (ordenados)')
print(f'  FORM_*:   {len(all_form_tokens)} (ordenados)')
print(f'  GOALS_*:  {len(all_goals_tokens)} (ordenados)')
print(f'  EVENT_*:  {len(all_event_tokens)}')
print(f'  STAGE_*:  {len(all_stage_tokens)}')

---
## Familia 1 — Vecindades puras: qué entendió el modelo de cada token

In [ ]:
# Helper para mostrar vecinos bonito
def show_neighbours(query, restrict, k=8, label=None):
    if not vocab.has(query):
        print(f'\n[{query}] NO está en el vocab.')
        return
    label = label or query
    nn = top_k_neighbours(model, vocab, query, k=k, restrict_to=restrict, exclude_self=True)
    print(f'\n[{label}]  top-{k}:')
    for tok, sim in nn:
        print(f'    {sim:+.4f}   {tok}')

# Equipos: los 6 candidatos al título
for q in ['TEAM_ARGENTINA', 'TEAM_BRAZIL', 'TEAM_FRANCE', 'TEAM_SPAIN', 'TEAM_ENGLAND', 'TEAM_GERMANY']:
    show_neighbours(q, restrict=all_team_tokens, k=6)

In [ ]:
# Equipos sin restricción: ¿en qué "familia" cae cada uno cuando puede ser cualquier cosa?
for q in ['TEAM_ARGENTINA', 'TEAM_FRANCE']:
    show_neighbours(q, restrict=None, k=10, label=f'{q} (TODOS los tokens)')

In [ ]:
# Jugadores: Messi (5503), Mbappé (3009), y algunos más conocidos
# (PLAYER ids fueron asignados según frecuencia en StatsBomb)
PLAYER_QUERIES = {
    'PLAYER_5503': 'Messi',
    'PLAYER_3009': 'Mbappé',
    'PLAYER_5487': 'Griezmann',
    'PLAYER_5485': 'Varane',
    'PLAYER_5477': 'Dembélé',
    'PLAYER_5476': 'Pavard',
}
for tok, name in PLAYER_QUERIES.items():
    show_neighbours(tok, restrict=all_player_tokens, k=6, label=f'{tok} ({name})')

In [ ]:
# Eventos
for q in all_event_tokens[:6]:
    show_neighbours(q, restrict=all_event_tokens, k=5)

# Stages
print('\n\n=== STAGES ===')
for q in all_stage_tokens:
    show_neighbours(q, restrict=all_stage_tokens, k=4)

---
## Familia 2 — Ordinalidad emergente en TODOS los buckets numéricos

Confirmamos el hallazgo principal del paper: los buckets cuantitativos están ordenados aunque hayan sido entrenados como categóricos.

In [ ]:
def ordinality_score(family_tokens):
    """Para cada token de la familia (ordenado), computa la similitud coseno
    contra todos los demás. Devuelve una matriz (N, N)."""
    n = len(family_tokens)
    M = np.zeros((n, n))
    for i, ti in enumerate(family_tokens):
        for j, tj in enumerate(family_tokens):
            if i == j:
                M[i, j] = 1.0
            else:
                ei = get_token_embedding(model, vocab, ti)
                ej = get_token_embedding(model, vocab, tj)
                M[i, j] = float(torch.nn.functional.cosine_similarity(
                    ei.unsqueeze(0), ej.unsqueeze(0)).item())
    return M

# ELO buckets: matriz completa
elo_matrix = ordinality_score(all_elo_tokens)
form_matrix = ordinality_score(all_form_tokens)
goals_matrix = ordinality_score(all_goals_tokens)

print('--- Matriz coseno ELO buckets ---')
df_elo = pd.DataFrame(elo_matrix,
                       index=[t.replace('ELO_BUCKET_', '') for t in all_elo_tokens],
                       columns=[t.replace('ELO_BUCKET_', '') for t in all_elo_tokens])
print(df_elo.round(2).to_string())

In [ ]:
# Visualizar las 3 matrices como heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, M, labels, title in zip(
    axes,
    [elo_matrix, form_matrix, goals_matrix],
    [[t.replace('ELO_BUCKET_', '') for t in all_elo_tokens],
     [t.replace('FORM_', '') for t in all_form_tokens],
     [t.replace('GOALS_', '') for t in all_goals_tokens]],
    ['ELO buckets', 'FORM buckets', 'GOALS buckets'],
):
    im = ax.imshow(M, cmap='RdYlBu_r', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_title(f'Cosine sim — {title}')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'embeddings_ordinality_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

# Cuantificar la monotonicidad: para cada bucket i, la similitud con el vecino
# distancia +1 debería ser mayor que con distancia +2, etc.
def monotonicity_score(M):
    """Fracción de tripletas (i, j, k) con i<j<k tal que M[i,j] > M[i,k]."""
    n = M.shape[0]
    correct = 0
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                total += 1
                if M[i, j] > M[i, k]:
                    correct += 1
    return correct / total if total > 0 else 0.0

print(f'\n=== Score de monotonicidad (random baseline = 0.50) ===')
print(f'ELO buckets:   {monotonicity_score(elo_matrix):.3f}')
print(f'FORM buckets:  {monotonicity_score(form_matrix):.3f}')
print(f'GOALS buckets: {monotonicity_score(goals_matrix):.3f}')

**Interpretación:** un score de **1.0** significa que para cada bucket de referencia, los vecinos están perfectamente ordenados por distancia numérica. **0.50** es lo que daría un modelo aleatorio. Esperamos ELO ≥ 0.85.

---
## Familia 3 — Estructura competitiva: centroides por confederación

In [ ]:
CONFEDERATIONS = {
    'CONMEBOL': ['Argentina', 'Brazil', 'Uruguay', 'Colombia', 'Chile', 'Peru', 'Ecuador',
                 'Paraguay', 'Bolivia', 'Venezuela'],
    'UEFA':     ['France', 'Germany', 'Spain', 'Italy', 'England', 'Portugal',
                 'Netherlands', 'Belgium', 'Croatia', 'Poland', 'Denmark', 'Sweden',
                 'Switzerland'],
    'CAF':      ['Nigeria', 'Egypt', 'Senegal', 'Morocco', 'Cameroon', 'Ghana',
                 'Algeria', 'Tunisia', 'Ivory Coast', 'South Africa'],
    'AFC':      ['Japan', 'South Korea', 'Iran', 'Saudi Arabia', 'Australia', 'Qatar', 'Iraq'],
    'CONCACAF': ['Mexico', 'United States', 'Canada', 'Costa Rica', 'Honduras',
                 'Panama', 'Jamaica'],
}

def slug(t):
    return f'TEAM_{t.upper().replace(" ", "_")}'

centroids = {}
for conf, teams in CONFEDERATIONS.items():
    tokens = [slug(t) for t in teams if vocab.has(slug(t))]
    if len(tokens) >= 3:
        centroids[conf] = cluster_centroid(model, vocab, tokens, normalize=True)

# Matriz de similitud entre centroides
from itertools import combinations
print('=== Similitud coseno entre centroides de confederaciones ===')
print(f'{"":12s}', '  '.join(f'{c:9s}' for c in centroids))
for c1 in centroids:
    row = f'{c1:12s}'
    for c2 in centroids:
        sim = torch.nn.functional.cosine_similarity(
            centroids[c1].unsqueeze(0), centroids[c2].unsqueeze(0)).item()
        row += f'  {sim:+.4f}'
    print(row)

In [ ]:
# Para cada equipo, calcular su distancia al centroide UEFA y CONMEBOL
# → ¿Cuál es el equipo sudamericano más "europeo"?
from torch.nn.functional import cosine_similarity

uefa_c = centroids.get('UEFA')
conmebol_c = centroids.get('CONMEBOL')

rows = []
for conf, teams in CONFEDERATIONS.items():
    for t in teams:
        tok = slug(t)
        if not vocab.has(tok): continue
        e = get_token_embedding(model, vocab, tok)
        e_n = e / (e.norm() + 1e-12)
        rows.append({
            'team': t,
            'real_conf': conf,
            'sim_to_UEFA':     float(cosine_similarity(e_n.unsqueeze(0), uefa_c.unsqueeze(0)).item()),
            'sim_to_CONMEBOL': float(cosine_similarity(e_n.unsqueeze(0), conmebol_c.unsqueeze(0)).item()),
        })
df = pd.DataFrame(rows)
df['Δ (UEFA - CONMEBOL)'] = df['sim_to_UEFA'] - df['sim_to_CONMEBOL']

print('\nLos 5 sudamericanos más "europeos" (mayor sim_UEFA − sim_CONMEBOL):')
print(df[df.real_conf == 'CONMEBOL'].sort_values('Δ (UEFA - CONMEBOL)', ascending=False).head(5).to_string(index=False))

print('\nLos 5 europeos más "sudamericanos":')
print(df[df.real_conf == 'UEFA'].sort_values('Δ (UEFA - CONMEBOL)').head(5).to_string(index=False))

---
## Familia 4 — Diagnóstico geométrico: por qué fallan las analogías

In [ ]:
# Normas por familia
FAMILIES = {
    'team': 'TEAM_',
    'player': 'PLAYER_',
    'elo_bucket': 'ELO_BUCKET_',
    'form': 'FORM_',
    'goals': 'GOALS_',
    'tournament': 'TOURNAMENT_',
    'event': 'EVENT_',
    'stage': 'STAGE_',
    'venue': 'VENUE_',
    'result': 'RESULT_',
    'score': 'SCORE_',
    'special': '[',
}
norm_stats = family_norm_stats(model, vocab, FAMILIES)
df_norms = pd.DataFrame(norm_stats).T
df_norms = df_norms[df_norms['n'] > 0].sort_values('mean_norm', ascending=False)
print('=== Norma L2 media de embeddings por familia ===')
print(df_norms.round(3).to_string())

print('\nInterpretación: las familias con norma más alta DOMINAN cuando se hace aritmética')
print('(B - A + C). Esto explica el sesgo de las analogías hacia familias con norma grande.')

In [ ]:
# Anisotropía intra-familia: ¿qué tan "colapsado" está el cono de cada familia?
print('=== Similitud coseno media INTRA-familia (anisotropía) ===')
print('Valores altos = la familia vive en un cono estrecho')
print('Valores cerca de 0 = familia bien distribuida en el espacio\n')
print(f'{"familia":<12} {"n_pairs":<10} {"mean":<10} {"median":<10} {"p25":<10} {"p75":<10}')
for name, prefix in [('team', 'TEAM_'), ('player', 'PLAYER_'),
                      ('elo', 'ELO_BUCKET_'), ('form', 'FORM_'),
                      ('tournament', 'TOURNAMENT_'), ('event', 'EVENT_')]:
    d = intra_family_cosine_distribution(model, vocab, prefix, sample_size=150)
    print(f'{name:<12} {d["n_pairs"]:<10d} {d["mean"]:<10.4f} {d["median"]:<10.4f} {d["p25"]:<10.4f} {d["p75"]:<10.4f}')

---
## Familia 5 — Comparación de tres técnicas de analogía

Probamos las mismas 6 analogías con cada técnica y comparamos.

In [ ]:
ANALOGIES = [
    ('TEAM_ARGENTINA', 'PLAYER_5503', 'TEAM_FRANCE',    'Argentina:Messi :: Francia:?'),
    ('TEAM_ARGENTINA', 'PLAYER_5503', 'TEAM_PORTUGAL',  'Argentina:Messi :: Portugal:?'),
    ('TEAM_FRANCE',    'PLAYER_3009', 'TEAM_ENGLAND',   'Francia:Mbappé :: Inglaterra:?'),
    ('TEAM_FRANCE',    'PLAYER_3009', 'TEAM_BRAZIL',    'Francia:Mbappé :: Brasil:?'),
    ('TEAM_BRAZIL',    'PLAYER_5503', 'TEAM_ARGENTINA', 'Brasil:Messi(!) :: Argentina:? (control)'),
    ('PLAYER_5503',    'TEAM_ARGENTINA', 'PLAYER_3009', 'Messi:Argentina :: Mbappé:?'),
]

def show_three_methods(a, b, c, desc, restrict, k=3):
    print(f'\n{"─"*70}')
    print(f'[{desc}]')
    print(f'{"─"*70}')
    
    # A. Aditivo sin restricción (baseline original)
    r1 = analogy_query(model, vocab, a, b, c, k=k, restrict_to=None)
    print(f'  Mikolov SIN restricción:')
    for t, s in r1:
        print(f'    {s:+.4f}   {t}')
    
    # B. Aditivo con restricción + normalize
    r2 = analogy_query(model, vocab, a, b, c, k=k, restrict_to=restrict, normalize=True)
    print(f'  Mikolov + L2-normalize + restrict:')
    for t, s in r2:
        print(f'    {s:+.4f}   {t}')
    
    # C. 3CosMul + restricción
    r3 = analogy_query_3cosmul(model, vocab, a, b, c, k=k, restrict_to=restrict)
    print(f'  3CosMul + restrict (Levy & Goldberg 2014):')
    for t, s in r3:
        print(f'    {s:.4f}   {t}')

for a, b, c, desc in ANALOGIES:
    if not (vocab.has(a) and vocab.has(b) and vocab.has(c)):
        print(f'\n[{desc}] tokens incompletos')
        continue
    # Las analogías Argentina:Messi :: Francia:? buscan UN JUGADOR como respuesta
    # Las analogías Messi:Argentina :: Mbappé:? buscan UN EQUIPO
    if 'PLAYER' in b:   # respuesta esperada: jugador
        restrict = all_player_tokens
    else:               # respuesta esperada: equipo
        restrict = all_team_tokens
    show_three_methods(a, b, c, desc, restrict)

---
## Familia 6 — Visualización 2D combinada

Proyectamos juntos: equipos top + buckets numéricos. Esto muestra **dos manifolds distintos** dentro del espacio.

In [ ]:
# Mezcla: 20 equipos top + buckets ELO + buckets FORM
TOP_TEAMS = ['Argentina', 'Brazil', 'France', 'Germany', 'Spain', 'Italy',
             'England', 'Portugal', 'Netherlands', 'Belgium', 'Uruguay',
             'Colombia', 'Croatia', 'Mexico', 'United States', 'Japan',
             'Morocco', 'Senegal', 'Australia', 'Switzerland']
team_toks = [slug(t) for t in TOP_TEAMS if vocab.has(slug(t))]

mixed_tokens = team_toks + all_elo_tokens + all_form_tokens
mixed_red = pca_2d(model, vocab, mixed_tokens)

fig, ax = plt.subplots(figsize=(13, 9))
for tok, xy in zip(mixed_red.tokens, mixed_red.coords):
    if tok.startswith('TEAM_'):
        ax.scatter(xy[0], xy[1], s=80, c='steelblue', edgecolors='black', alpha=0.7)
        ax.annotate(tok.replace('TEAM_', ''), xy=xy, fontsize=7, xytext=(3, 3), textcoords='offset points')
    elif tok.startswith('ELO_BUCKET_'):
        ax.scatter(xy[0], xy[1], s=120, c='orange', marker='s', edgecolors='black', alpha=0.8)
        ax.annotate(tok.replace('ELO_BUCKET_', 'E'), xy=xy, fontsize=8, ha='center', va='center', fontweight='bold')
    elif tok.startswith('FORM_'):
        ax.scatter(xy[0], xy[1], s=120, c='green', marker='^', edgecolors='black', alpha=0.8)
        ax.annotate(tok.replace('FORM_', 'F_'), xy=xy, fontsize=7, ha='center', va='center')

ax.set_title(f'PCA: equipos (azul) + ELO buckets (naranja) + FORM buckets (verde)\n'
             f'Var: {mixed_red.explained_variance[0]:.0%}, {mixed_red.explained_variance[1]:.0%}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'embeddings_mixed_pca.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Resumen — para el paper

Llenar al final:

**Vecindades puras:**
- [ ] ¿Los 6 candidatos al título tienen vecinos coherentes? (sí/no)
- [ ] Algún resultado sorprendente? _____

**Ordinalidad:**
- [ ] Score monotonicidad ELO: _____
- [ ] Score monotonicidad FORM: _____
- [ ] Score monotonicidad GOALS: _____

**Estructura competitiva:**
- [ ] Confederación más cercana a UEFA: _____
- [ ] Selección sudamericana más "europea": _____
- [ ] Selección europea más "sudamericana": _____

**Diagnóstico geométrico:**
- [ ] Familia con mayor norma: _____ (norma = _____)
- [ ] Familia con mayor anisotropía intra (mean sim alta): _____

**Analogías — tasa de aciertos por método sobre las 6 pruebas:**
- [ ] Mikolov sin restricción: ___/6
- [ ] Mikolov + L2-norm + restrict: ___/6
- [ ] 3CosMul + restrict: ___/6
- [ ] ¿Salió Mbappé al menos UNA vez? _____

**Conclusión para el paper:** las analogías Mikolov son débiles en este corpus chico, pero la **ordinalidad** y los **vecinos puros** son robustos. El paper se sostiene sobre estos dos últimos.